# Feature Selection
This notebook is the continuation of the previous one about **Exploratory Data analysis** ([eda&feat_candidates.ipynb](./eda&feat_candidates.ipynb)).
This notebook shows a feature selection using **test with machine learning models**.

In [1]:
import mlflow
from dotenv import load_dotenv

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("rice_price_product_experiments")
load_dotenv()

True

In [181]:
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, TimeSeriesSplit, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import numpy as np
import statsmodels.api as sm
import itertools
from datetime import datetime
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [55]:
rice_data = pd.read_csv("./data/data_warehouse/candidates_features.csv")

In [56]:
candidate_features = ['usdmga', 'market_id', 'DAP', 'latitude']
target = ["price_transformed"]

# Process the target as non time-serie

I want to compare all the possible combination of the candidate features to see which one gives the best performance.

In [57]:
def all_combinations(iterable):
    "Generates all combinations of an iterable (powerset, excluding empty set)"
    s = list(iterable)
    return itertools.chain.from_iterable(itertools.combinations(s, r) for r in range(1, len(s) + 1))

In [58]:
all_combs = list(all_combinations(candidate_features))

Here are the metrics used to compare the performance of the model.

In [59]:
metrics = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']

In [75]:
def mean_scores(scores_lists):
    scores_lists = pd.DataFrame(scores_lists)
    scores_lists[['mse', 'train_mse', 'mae', 'train_mae']] = scores_lists[['test_neg_mean_squared_error', 'train_neg_mean_squared_error', 'test_neg_mean_absolute_error', 'train_neg_mean_absolute_error']].abs()
    scores_lists["rmse"] = np.sqrt(scores_lists["test_neg_mean_squared_error"].abs())
    scores_lists["train_rmse"] = np.sqrt(scores_lists["train_neg_mean_squared_error"].abs())
    return scores_lists.mean().to_dict()

Next, I implement a Dummy model that will be used as **baseline model**.

In [76]:
Baseline_model = DummyRegressor(strategy="mean")
with mlflow.start_run(run_name=f"Baseline_Model_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}"):
    baseline_scores = cross_validate(Baseline_model, rice_data[candidate_features], rice_data[target], scoring=metrics, cv=5, return_train_score=True)
    mlflow.log_metrics(mean_scores(baseline_scores))
    mlflow.set_tag("features", ", ".join(candidate_features))

🏃 View run Baseline_Model_2026-03-30_15:09:42 at: http://localhost:5000/#/experiments/6/runs/6072e65c0660441cb96c25ef83f81bb7
🧪 View experiment at: http://localhost:5000/#/experiments/6


In [77]:
with mlflow.start_run(run_name=f"Lin_Reg_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in all_combs:
        X = rice_data[list(feature)]
        y = rice_data[target]
        model = LinearRegression()
        with mlflow.start_run(run_name=f"Lin_reg_{i}", nested=True) as run:
            scores = cross_validate(model, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Lin_reg_0 at: http://localhost:5000/#/experiments/6/runs/57d374fac118447c957f4500b30342ab
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_1 at: http://localhost:5000/#/experiments/6/runs/28af800a47404d019da9c3bbefce69ab
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_2 at: http://localhost:5000/#/experiments/6/runs/50068791a2e24c90baf201dcf1bf4b68
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_3 at: http://localhost:5000/#/experiments/6/runs/3f3614b4e3684e749844c8a54f26aa1e
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_4 at: http://localhost:5000/#/experiments/6/runs/3cbc26f9118e43509d23028562198998
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_5 at: http://localhost:5000/#/experiments/6/runs/91ccde8c256c41faacf2d71469b4d513
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_6 at: http://lo

In [78]:
with mlflow.start_run(run_name=f"Lin_Reg_Interact_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in all_combs:
        X = rice_data[list(feature)]
        y = rice_data[target]
        pipeline = make_pipeline(
            PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
            LinearRegression()
        )
        with mlflow.start_run(run_name=f"Lin_reg_{i}", nested=True) as run:
            scores = cross_validate(pipeline, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Lin_reg_0 at: http://localhost:5000/#/experiments/6/runs/b19625875eb3416984a0d11c2ee00360
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_1 at: http://localhost:5000/#/experiments/6/runs/d2fa08836b6444b7b7d4d42bbe31f51e
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_2 at: http://localhost:5000/#/experiments/6/runs/be3ec4cd566c46339bd18a5dd06b9dc7
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_3 at: http://localhost:5000/#/experiments/6/runs/5e214bcaafea4b2888f31b1012b25130
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_4 at: http://localhost:5000/#/experiments/6/runs/1522ca0eef4e451a92b84abcf07444d1
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_5 at: http://localhost:5000/#/experiments/6/runs/1b2391c6df464097875f44812b1a6977
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg_6 at: http://lo

In [ ]:
with mlflow.start_run(run_name=f"Ridge_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in all_combs:
        X = rice_data[list(feature)]
        y = rice_data[target]
        model = Ridge(alpha=1.0)
        with mlflow.start_run(run_name=f"Ridge_{i}", nested=True) as run:
            scores = cross_validate(model, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Ridge_0 at: http://localhost:5000/#/experiments/6/runs/97ab34815f1f43b096cb1e5f8f15ab92
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_1 at: http://localhost:5000/#/experiments/6/runs/1c58c40d1d364eceb57eee71ee99288f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_2 at: http://localhost:5000/#/experiments/6/runs/445fe1eb469442f98c4b17edf38dbba8
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_3 at: http://localhost:5000/#/experiments/6/runs/7b6f6d543502416ea5dd63ea9224d267
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_4 at: http://localhost:5000/#/experiments/6/runs/1d7043a9dd8c4c8a8237961c1501b568
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_5 at: http://localhost:5000/#/experiments/6/runs/a7d9c44421e1473791baf187213ca17b
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_6 at: http://localhost:5000/#

In [83]:
with mlflow.start_run(run_name=f"Lasso_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in all_combs:
        X = rice_data[list(feature)]
        y = rice_data[target]
        model = Lasso(alpha=1.0)
        with mlflow.start_run(run_name=f"Lasso_{i}", nested=True) as run:
            scores = cross_validate(model, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Lasso_0 at: http://localhost:5000/#/experiments/6/runs/0e5ab299b247495aa078d50f7449f9a4
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_1 at: http://localhost:5000/#/experiments/6/runs/527e54a8638747adb6acf4f964e341d2
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_2 at: http://localhost:5000/#/experiments/6/runs/1811a0806b7f49c8a9b2edbefbf9a051
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_3 at: http://localhost:5000/#/experiments/6/runs/f2e37e577cdd426ea19fb79add1c1245
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_4 at: http://localhost:5000/#/experiments/6/runs/ad427037876645df9eeac9cada0a3657
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_5 at: http://localhost:5000/#/experiments/6/runs/265f49ae9b394346a1c6eab93b9b8fcb
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_6 at: http://localhost:5000/#

In [137]:
rice_data_applicants = pd.read_csv("./data/data_warehouse/applicants_features.csv", index_col=0)

In [90]:
applicant_combs = list(all_combinations(['DAP', 'market_id', 'gasoline_price', 'diesel_price', 'usdmga']))

In [93]:
with mlflow.start_run(run_name=f"Lin_Reg_Applicants_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicant_combs:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        model = LinearRegression()
        with mlflow.start_run(run_name=f"Lin_reg__app_{i}", nested=True) as run:
            scores = cross_validate(model, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Lin_reg__app_0 at: http://localhost:5000/#/experiments/6/runs/f747c6fccf9d43caba08ba834194090b
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_1 at: http://localhost:5000/#/experiments/6/runs/49d1f500c37c4881b0ec4c499ac8c9ee
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_2 at: http://localhost:5000/#/experiments/6/runs/a3ef4fe3be0a4233bb761afe05dd9316
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_3 at: http://localhost:5000/#/experiments/6/runs/dd85b5a594c14ee0a489cf77ed9032f0
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_4 at: http://localhost:5000/#/experiments/6/runs/ca64acf4127c400fb11986c83451f5a3
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_5 at: http://localhost:5000/#/experiments/6/runs/e19c245090104f33a7b1a328b8c412e9
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 Vi

In [94]:
with mlflow.start_run(run_name=f"Lin_Reg_Applicants_interact_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicant_combs:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        pipeline = make_pipeline(
            PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
            LinearRegression()
        )
        with mlflow.start_run(run_name=f"Lin_reg__app_interact_{i}", nested=True) as run:
            scores = cross_validate(pipeline, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Lin_reg__app_interact_0 at: http://localhost:5000/#/experiments/6/runs/0472d1522eeb4baf82e9fe5342292b61
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_interact_1 at: http://localhost:5000/#/experiments/6/runs/0d25e191f1e64d8a9a2d7bcdbe064fe2
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_interact_2 at: http://localhost:5000/#/experiments/6/runs/d8f20fbbf46641e892c03a87eff1d1d1
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_interact_3 at: http://localhost:5000/#/experiments/6/runs/ed9f8f1625ec4c5a91403f0c69c3d710
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_interact_4 at: http://localhost:5000/#/experiments/6/runs/85c959ec43bb441c9a9e7bfc5eb69f3a
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_reg__app_interact_5 at: http://localhost:5000/#/experiments/6/runs/f9c4ce5b2f1445bdb7b36483e30ee10b
🧪 View exp

In [95]:
with mlflow.start_run(run_name=f"Ridge_Applicants_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicant_combs:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        model = Ridge(alpha=1.0)
        with mlflow.start_run(run_name=f"Rigde__app_{i}", nested=True) as run:
            scores = cross_validate(model, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Rigde__app_0 at: http://localhost:5000/#/experiments/6/runs/d66a3451423e42d2ac3c25dfb4c3b13c
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Rigde__app_1 at: http://localhost:5000/#/experiments/6/runs/a93935f220384ad3a894dd091985ce0a
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Rigde__app_2 at: http://localhost:5000/#/experiments/6/runs/465da882a95f40f09809ba00f1851b0f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Rigde__app_3 at: http://localhost:5000/#/experiments/6/runs/63198452ab5a4ed193db5497574ec546
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Rigde__app_4 at: http://localhost:5000/#/experiments/6/runs/9d22423206b84e0d9822c080f02f7d8d
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Rigde__app_5 at: http://localhost:5000/#/experiments/6/runs/13180fa6d48d467ba389998a3532dfb0
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Rigde

In [96]:
with mlflow.start_run(run_name=f"Lasso_Applicants_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicant_combs:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        model = Lasso(alpha=1.0)
        with mlflow.start_run(run_name=f"Lasso__app_{i}", nested=True) as run:
            scores = cross_validate(model, X, y, scoring=metrics, cv=5, return_train_score=True)
            mlflow.log_metrics(mean_scores(scores))
            mlflow.set_tag("features", ", ".join(list(feature)))
        i += 1

🏃 View run Lasso__app_0 at: http://localhost:5000/#/experiments/6/runs/ad69becf7a584b9793c6a0deddb9102f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_1 at: http://localhost:5000/#/experiments/6/runs/7d0e2b7a5f12435e9da1914c4d010848
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_2 at: http://localhost:5000/#/experiments/6/runs/c8dcdc20c7da4ff19a98a48a257df2ca
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_3 at: http://localhost:5000/#/experiments/6/runs/f3e00c6372ae48ec9c452d830faab0c0
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_4 at: http://localhost:5000/#/experiments/6/runs/c2fefc58ff974c948533964bff52623e
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_5 at: http://localhost:5000/#/experiments/6/runs/2c8b5c356d344ed19719d339513b0d0f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso

/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.167e+02, tolerance: 1.987e-01
  model = cd_fast.enet_coordinate_descent(


🏃 View run Lasso__app_12 at: http://localhost:5000/#/experiments/6/runs/e37d6956e99f44408d43ece05c00c85c
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_13 at: http://localhost:5000/#/experiments/6/runs/a394e0b9ff184571a4c508101a413795
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_14 at: http://localhost:5000/#/experiments/6/runs/28b29655bc464e3b9cb62c64b04971d2
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_15 at: http://localhost:5000/#/experiments/6/runs/042bb81b4534421686b9f892697b45fa
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_16 at: http://localhost:5000/#/experiments/6/runs/82586f95c0d54d71b0188f105610fe7f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_17 at: http://localhost:5000/#/experiments/6/runs/d0bceeeb82e8469f94d0e0450bd91669
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.400e+02, tolerance: 1.987e-01
  model = cd_fast.enet_coordinate_descent(


🏃 View run Lasso__app_18 at: http://localhost:5000/#/experiments/6/runs/70b005b1573d4238b19852d414162f4e
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_19 at: http://localhost:5000/#/experiments/6/runs/2af6f03b83ff40afba6fcd099b1628c9
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_20 at: http://localhost:5000/#/experiments/6/runs/93f4290c30274230b4a94bcb3857bab2
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_21 at: http://localhost:5000/#/experiments/6/runs/c3f1714df5be4b53830809181671ff39
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.085e+02, tolerance: 1.987e-01
  model = cd_fast.enet_coordinate_descent(


🏃 View run Lasso__app_22 at: http://localhost:5000/#/experiments/6/runs/7dff39942d284f81a345dadcbc4538ae
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_23 at: http://localhost:5000/#/experiments/6/runs/0eb3ee1f1ff443e28e8174ef2e54d92d
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_24 at: http://localhost:5000/#/experiments/6/runs/dfa3b508f3e94fc6b0db4275687e1865
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_25 at: http://localhost:5000/#/experiments/6/runs/396c94ceb0f245c2b199593744e96702
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.068e+01, tolerance: 1.987e-01
  model = cd_fast.enet_coordinate_descent(


🏃 View run Lasso__app_26 at: http://localhost:5000/#/experiments/6/runs/4342b28f814840658e801a1cdd12e279
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_27 at: http://localhost:5000/#/experiments/6/runs/e955b80013d54f8da32461ed2e1431b7
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_28 at: http://localhost:5000/#/experiments/6/runs/22128b16174942fcb187efb4c2149aea
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_29 at: http://localhost:5000/#/experiments/6/runs/56bf1137d9cb4216be6aae16ca2afb55
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso__app_30 at: http://localhost:5000/#/experiments/6/runs/48d39a2f9514416886a8e6ce1fc78d5f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_Applicants_2026-03-30_15:36:12 at: http://localhost:5000/#/experiments/6/runs/5f81c9e450db4729b67713e7e76b249a
🧪 View experiment at: http://localhost:5000/#/e

In [130]:
applicants_with_cat = list(all_combinations(['gasoline_price', 'diesel_price', 'usdmga', "admin1" , "pricetype", "commodity", "month"]))
rice_data_applicants['month'] = rice_data_applicants["month"].astype(str)

In [134]:
with mlflow.start_run(run_name=f"Lin_Reg_cats_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = LinearRegression()
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Lin_Reg_cats_{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=5, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Lin_Reg_cats_0 at: http://localhost:5000/#/experiments/6/runs/db6526edf0e849b3af1e9567d31ee029
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_1 at: http://localhost:5000/#/experiments/6/runs/44f88ac49b1842fe9433a44a0d55cb62
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_2 at: http://localhost:5000/#/experiments/6/runs/aacbf491008f4c12b3309139084d2792
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_3 at: http://localhost:5000/#/experiments/6/runs/d48bd5e847ff47f58ba333830751df94
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_4 at: http://localhost:5000/#/experiments/6/runs/70aabbe724ef452d92eaba6d14b534a1
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_5 at: http://localhost:5000/#/experiments/6/runs/07b99f896ce24a81806c7ae89c2d311f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 Vi

In [183]:
alpha_value = 1
random_state = 42

In [184]:
with mlflow.start_run(run_name=f"LinReg_cats_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = LinearRegression()
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"LinReg_cats_{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.log_params({"random_state": random_state})
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run LinReg_cats_0 at: http://localhost:5000/#/experiments/6/runs/a066299bfecb46769728246d4e8f7ff7
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run LinReg_cats_1 at: http://localhost:5000/#/experiments/6/runs/764a8666ea5a41ddbe2b71e33a028981
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run LinReg_cats_2 at: http://localhost:5000/#/experiments/6/runs/5e1ff99da4084db29443e0d0e5360abf
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run LinReg_cats_3 at: http://localhost:5000/#/experiments/6/runs/1c9b18ed82144a3980d9f92a2e4d0a34
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run LinReg_cats_4 at: http://localhost:5000/#/experiments/6/runs/6ea726b7577f48be9c34a2342dd6800c
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run LinReg_cats_5 at: http://localhost:5000/#/experiments/6/runs/2bd524ae33cb44c4b292ffff1fcff59f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run

In [185]:
with mlflow.start_run(run_name=f"Ridge_cats_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = Ridge(alpha=alpha_value)
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Ridge_cats_{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.log_params({"alpha": alpha_value})
                mlflow.log_params({"random_state": random_state})
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Ridge_cats_0 at: http://localhost:5000/#/experiments/6/runs/263b6872b315451e9f5ce292714a453a
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_1 at: http://localhost:5000/#/experiments/6/runs/b3ea9f08a872458cb8249877f21704be
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_2 at: http://localhost:5000/#/experiments/6/runs/1686e00debf5405f9f32fd1d72bc8b6d
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_3 at: http://localhost:5000/#/experiments/6/runs/232a18a8c63c4acfb2a2ce7fa47fd70d
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_4 at: http://localhost:5000/#/experiments/6/runs/ccec971617cb44439ce54d0da8639799
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_5 at: http://localhost:5000/#/experiments/6/runs/e6991ddb5d6540f39e1cb15962cecba7
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge

In [186]:
with mlflow.start_run(run_name=f"Lasso_cats_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = Lasso(alpha=alpha_value)
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Lasso_cats_{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.log_params({"alpha": alpha_value})
                mlflow.log_params({"random_state": random_state})
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Lasso_cats_0 at: http://localhost:5000/#/experiments/6/runs/7d60fb236548434ab1bb2a84da1a4e77
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_1 at: http://localhost:5000/#/experiments/6/runs/70ed88e8074e4821b5cadf2870ad73ee
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_2 at: http://localhost:5000/#/experiments/6/runs/69bf7c5d6f7f4610b6a2ccfa417ec0bd
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_3 at: http://localhost:5000/#/experiments/6/runs/01606adac47c4df88bafcff5664c13e8
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_4 at: http://localhost:5000/#/experiments/6/runs/40ded2b240a44725ae5accb0a117d0d4
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_5 at: http://localhost:5000/#/experiments/6/runs/db8b9eeb7b0741a1ac728ab65360bcf5
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso

# Standardized data

In [187]:
with mlflow.start_run(run_name=f"Lin_Reg_cats_std_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        pipeline = make_pipeline(StandardScaler(), LinearRegression())
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Lin_Reg_cats_std_{i}", nested=True) as run:
                scores = cross_validate(pipeline, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
                mlflow.log_params({"random_state": random_state})
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Lin_Reg_cats_std_0 at: http://localhost:5000/#/experiments/6/runs/81364a8ef25a4f1688b599cd78da26fc
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_std_1 at: http://localhost:5000/#/experiments/6/runs/c3b786a39ce24699b929b49b488c85c6
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_std_2 at: http://localhost:5000/#/experiments/6/runs/146ac6d5aef34d1481998400e073c6c6
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_std_3 at: http://localhost:5000/#/experiments/6/runs/5e9de6d29b294238a893ff21a7a80b7c
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_std_4 at: http://localhost:5000/#/experiments/6/runs/f1e7f65f0ff046178126175368a03382
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_cats_std_5 at: http://localhost:5000/#/experiments/6/runs/99e11ea72b79496bb9196897d49798c7
🧪 View experiment at: http://localhost:5

In [188]:
with mlflow.start_run(run_name=f"Ridge_cats_std_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        pipeline = make_pipeline(StandardScaler(), Ridge(alpha=alpha_value))
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Ridge_cats_std_{i}", nested=True) as run:
                scores = cross_validate(pipeline, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
                mlflow.log_params({"alpha": alpha_value})
                mlflow.log_params({"random_state": random_state})
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Ridge_cats_std_0 at: http://localhost:5000/#/experiments/6/runs/fb92d281a1e2451596f2f784b155e8d5
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_std_1 at: http://localhost:5000/#/experiments/6/runs/38fa82f0c44b41bda7abe9efdec07663
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_std_2 at: http://localhost:5000/#/experiments/6/runs/c28af55857e54b22aabdc4939b682b49
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_std_3 at: http://localhost:5000/#/experiments/6/runs/6ab3b097c8ff46108b22f4870a6b20eb
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_std_4 at: http://localhost:5000/#/experiments/6/runs/824c2cbc94fe4abc8859beea5e523341
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_cats_std_5 at: http://localhost:5000/#/experiments/6/runs/7dd36220d8a74d09ab216e594be68022
🧪 View experiment at: http://localhost:5000/#/experi

In [189]:
with mlflow.start_run(run_name=f"Lasso_cats_std_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        pipeline = make_pipeline(StandardScaler(), Lasso(alpha=alpha_value))
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Lasso_cats_std_{i}", nested=True) as run:
                scores = cross_validate(pipeline, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
                mlflow.log_params({"alpha": alpha_value})
                mlflow.log_params({"random_state": random_state})
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Lasso_cats_std_0 at: http://localhost:5000/#/experiments/6/runs/4f3d633b763a45449bd88e1b7c18a576
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_std_1 at: http://localhost:5000/#/experiments/6/runs/df3f8441401b43b982a3a9561bc61024
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_std_2 at: http://localhost:5000/#/experiments/6/runs/4177ac65f8a7491b843b94ab0ae0c717
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_std_3 at: http://localhost:5000/#/experiments/6/runs/e687e67a61204f588e52fd1e5524d928
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_std_4 at: http://localhost:5000/#/experiments/6/runs/1a1bd15e05f244e78f453700739af6c3
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_cats_std_5 at: http://localhost:5000/#/experiments/6/runs/0bb0f4cc3bd245b4b501371d7978f115
🧪 View experiment at: http://localhost:5000/#/experi

# Process the target as time serie

In [138]:
with mlflow.start_run(run_name=f"Lin_Reg_timeserie_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = LinearRegression()
        tscv = TimeSeriesSplit(n_splits=5)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Lin_Reg_timeserie_{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=tscv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Lin_Reg_timeserie_0 at: http://localhost:5000/#/experiments/6/runs/8b2b60b05e314157917c1d382a16f42a
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_timeserie_1 at: http://localhost:5000/#/experiments/6/runs/b058a9df290f46569798f7608ba8cf51
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_timeserie_2 at: http://localhost:5000/#/experiments/6/runs/97659dd9fb034b42b80778a126322e5e
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_timeserie_3 at: http://localhost:5000/#/experiments/6/runs/acdccf21cf234dc39e4dbc0175bb5732
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_timeserie_4 at: http://localhost:5000/#/experiments/6/runs/a941a4f92d0d4027b0ea2ad499a55c73
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lin_Reg_timeserie_5 at: http://localhost:5000/#/experiments/6/runs/7830961a58e349fb8f7186e2dcb5ab96
🧪 View experiment at: http://local

In [139]:
with mlflow.start_run(run_name=f"Ridge_timeserie_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = Ridge()
        tscv = TimeSeriesSplit(n_splits=5)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Ridge_timeserie_{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=tscv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Ridge_timeserie_0 at: http://localhost:5000/#/experiments/6/runs/c75faa5a99d649dcbd9540d072665e64
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_timeserie_1 at: http://localhost:5000/#/experiments/6/runs/62a0afc6c69a487cb2bb5d0275da5950
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_timeserie_2 at: http://localhost:5000/#/experiments/6/runs/8957d2dd825b451fac5f3bbc286dd21b
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_timeserie_3 at: http://localhost:5000/#/experiments/6/runs/dc87bc40aadc486cad25f5fca30867bb
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_timeserie_4 at: http://localhost:5000/#/experiments/6/runs/474759e2a0eb48dfafab9217717f5645
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Ridge_timeserie_5 at: http://localhost:5000/#/experiments/6/runs/f1c4d54c84c2447daf1cafa166b5384e
🧪 View experiment at: http://localhost:5000/#/

In [140]:
with mlflow.start_run(run_name=f"Lasso_timeserie_{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = Lasso()
        tscv = TimeSeriesSplit(n_splits=5)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"Lasso_timeserie_{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=tscv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run Lasso_timeserie_0 at: http://localhost:5000/#/experiments/6/runs/1d3e0cf58c5c4d3f8788468b0179854e
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_1 at: http://localhost:5000/#/experiments/6/runs/57513f18668c424fb5169452a47c7362
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_2 at: http://localhost:5000/#/experiments/6/runs/1d93798bea5443b198b957399cad309e
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_3 at: http://localhost:5000/#/experiments/6/runs/6b293c7767d940fb9ce08393c1d917f9
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_4 at: http://localhost:5000/#/experiments/6/runs/9f8a81aa59a14d38b2f51173c326c206
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_5 at: http://localhost:5000/#/experiments/6/runs/fc9122e23b8840cda8b166bfb434ac62
🧪 View experiment at: http://localhost:5000/#/

/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.935e+02, tolerance: 1.128e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.671e+02, tolerance: 1.455e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning

🏃 View run Lasso_timeserie_8 at: http://localhost:5000/#/experiments/6/runs/d6d66d69d0704931a9875e6d7d01c749
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_9 at: http://localhost:5000/#/experiments/6/runs/35eb12263ea94b6b82c504d320014b59
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_10 at: http://localhost:5000/#/experiments/6/runs/c83629492f74465898427d142eb6d76c
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_11 at: http://localhost:5000/#/experiments/6/runs/93fa2b0e9f9a42959c67413dcea8218c
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_12 at: http://localhost:5000/#/experiments/6/runs/b377df0fb904444eb8a14ac6f7d0a60c
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_13 at: http://localhost:5000/#/experiments/6/runs/511d7136882b4309ad90f4e8c8c01ef8
🧪 View experiment at: http://localhost:500

/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.226e+01, tolerance: 1.128e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.459e+01, tolerance: 1.455e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning

🏃 View run Lasso_timeserie_28 at: http://localhost:5000/#/experiments/6/runs/b2f7d5cdd59b443893d3823c64d843c5
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_29 at: http://localhost:5000/#/experiments/6/runs/c6a0fbf80426417a9219a1cf3bb08dfd
🧪 View experiment at: http://localhost:5000/#/experiments/6


...

/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.226e+01, tolerance: 1.128e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.459e+01, tolerance: 1.455e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning

🏃 View run Lasso_timeserie_122 at: http://localhost:5000/#/experiments/6/runs/270ca319af194f0e9ce44b789da2408b
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_123 at: http://localhost:5000/#/experiments/6/runs/1c67d3fff6d14f7e9536f6838b6c1cb6
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.935e+02, tolerance: 1.128e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.671e+02, tolerance: 1.455e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning

🏃 View run Lasso_timeserie_124 at: http://localhost:5000/#/experiments/6/runs/0dea56dfe3c24c69a75e6acc2efc6cd3
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_125 at: http://localhost:5000/#/experiments/6/runs/66addc414da14d8f9fc2fe49bb6f0151
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_126 at: http://localhost:5000/#/experiments/6/runs/1073b5aab70d4cb893f74ab65bab4a02
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run Lasso_timeserie_2026-03-30_17:08:30 at: http://localhost:5000/#/experiments/6/runs/9de0f984317d4555bb0921490ac9f4b8
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.226e+01, tolerance: 1.128e-01
  model = cd_fast.enet_coordinate_descent(
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.459e+01, tolerance: 1.455e-01
  model = cd_fast.enet_coordinate_descent(


# Try other models

In [190]:
with mlflow.start_run(run_name=f"DecisionTreeRegressor__{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = DecisionTreeRegressor()
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"DecisionTreeRegressor__{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
                mlflow.log_params({"random_state": random_state})
        else:
            print(X_encoded.columns)
        i += 1

🏃 View run DecisionTreeRegressor__0 at: http://localhost:5000/#/experiments/6/runs/62447fdb715b4fe59ffc103fb79c8e8f
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run DecisionTreeRegressor__1 at: http://localhost:5000/#/experiments/6/runs/329c8fa3a9c74aa39c4cc866819d162b
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run DecisionTreeRegressor__2 at: http://localhost:5000/#/experiments/6/runs/84db192ca8a94de99960d0529af339d7
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run DecisionTreeRegressor__3 at: http://localhost:5000/#/experiments/6/runs/9363171174b0448c8f417de253dd8eee
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run DecisionTreeRegressor__4 at: http://localhost:5000/#/experiments/6/runs/19c5d129234245488a2e03514e62a5c9
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run DecisionTreeRegressor__5 at: http://localhost:5000/#/experiments/6/runs/339cbbd7236a4a18a4a9bf41225e4056
🧪 Vi

In [191]:
with mlflow.start_run(run_name=f"RandomForestRegressor__{datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}") as parent_run:
    i = 0
    for feature in applicants_with_cat:
        X = rice_data_applicants[list(feature)]
        y = rice_data_applicants[target]
        X_encoded = pd.get_dummies(X, prefix="", prefix_sep="", dtype=int)
        X_numeric = X_encoded.select_dtypes(include=[np.number])
        model = RandomForestRegressor()
        KFcv = KFold(n_splits=5, shuffle=True, random_state=random_state)
        if X_numeric.shape[1] > 0:
            with mlflow.start_run(run_name=f"RandomForestRegressor__{i}", nested=True) as run:
                scores = cross_validate(model, X_numeric, y, scoring=metrics, cv=KFcv, return_train_score=True)
                mlflow.log_metrics(mean_scores(scores))
                mlflow.set_tag("features", ", ".join(list(X_numeric.columns)))
                mlflow.log_params({"random_state": random_state})
        else:
            print(X_encoded.columns)
        i += 1

/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/S

🏃 View run RandomForestRegressor__0 at: http://localhost:5000/#/experiments/6/runs/ac8ca8421c86480ead7a8a6029c5fdb1
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/S

🏃 View run RandomForestRegressor__1 at: http://localhost:5000/#/experiments/6/runs/dba01f8eb2454367a3dbd6f00746a063
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/S

...

🏃 View run RandomForestRegressor__124 at: http://localhost:5000/#/experiments/6/runs/a2b2e53371cc4c0eb01007d9e111e83a
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/S

🏃 View run RandomForestRegressor__125 at: http://localhost:5000/#/experiments/6/runs/43d0b9629fff4e3b84bd8dea3203936b
🧪 View experiment at: http://localhost:5000/#/experiments/6


/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/Study/Projects/full_project_learn/rice_price_predict/backend/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bx/S

🏃 View run RandomForestRegressor__126 at: http://localhost:5000/#/experiments/6/runs/0e72e3d9e0c54fe5ba017e87b38030d2
🧪 View experiment at: http://localhost:5000/#/experiments/6
🏃 View run RandomForestRegressor__2026-03-30_23:11:51 at: http://localhost:5000/#/experiments/6/runs/ca94d590e119481b9c9d4a3e021a0519
🧪 View experiment at: http://localhost:5000/#/experiments/6
